# Libraries

In [1]:
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo
from sklearn.datasets import fetch_california_housing

from tensorflow.keras import layers, models
import tensorflow as tf

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# own functions
from active_regression import random_sampling, uncertainty_sampling_reg, label_data, active_learning_reg, sasla_reg
from active_regression import tune_params, retrieve_params, cv_regression, create_regmodel

from time import time

# Models

## California Housing

In [10]:
california = fetch_california_housing(as_frame=True)
california_df = california.frame
california_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [11]:
X = california_df.drop(columns=['MedHouseVal']).to_numpy()
y = california_df.loc[:, 'MedHouseVal'].to_numpy()

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=51
)

In [5]:
acts = ['relu', 'leaky_relu', 'elu']
for activation in acts:
    base_params = tune_params(X_pool, y_pool,
                              loss='mse',
                              out_units=1, out_act='linear',
                              k=5, activation=activation,
                              random_state=42,
                              n_trials=80,
                              export=True, file = f'params/california/{activation}.json'
    )

[I 2026-09-25 07:37:05,168] A new study created in memory with name: no-name-595df385-cb03-48b6-b8c7-d5e3118a9b9b


[I 2026-09-25 07:41:21,891] Trial 0 finished with value: 0.3916826523948034 and parameters: {'hidden_layer': 21, 'learning_rate': 0.003625073723242179, 'momentum': 0.367834230369271, 'weight_decay': 2.5430405201770885e-05}. Best is trial 0 with value: 0.3916826523948034.
[I 2026-09-25 07:41:28,609] Trial 1 finished with value: inf and parameters: {'hidden_layer': 55, 'learning_rate': 0.0044267163631692774, 'momentum': 0.7719791575309446, 'weight_decay': 4.871099891275516e-05}. Best is trial 0 with value: 0.3916826523948034.
[I 2026-09-25 07:47:38,742] Trial 2 finished with value: 0.5027052022348896 and parameters: {'hidden_layer': 30, 'learning_rate': 0.00030453461462901546, 'momentum': 0.003086914225791774, 'weight_decay': 2.7796470503845208e-05}. Best is trial 0 with value: 0.3916826523948034.
[I 2026-09-25 07:51:41,132] Trial 3 finished with value: 0.35935990197524603 and parameters: {'hidden_layer': 80, 'learning_rate': 0.006751255703576771, 'momentum': 0.16903264886659605, 'weight

{'hidden_layer': 74, 'learning_rate': 0.0018151866586540834, 'momentum': 0.7973662107501709, 'weight_decay': 1.7969467048451465e-05}
0.3590877110713562


[I 2026-09-25 09:44:25,376] Trial 0 finished with value: 0.4570477371068279 and parameters: {'hidden_layer': 42, 'learning_rate': 0.00028896699348474427, 'momentum': 0.5612715360675683, 'weight_decay': 0.0014350985614902906, 'leaky_relu_alpha': 0.009315682944208856}. Best is trial 0 with value: 0.4570477371068279.
[I 2026-09-25 09:48:17,114] Trial 1 finished with value: inf and parameters: {'hidden_layer': 127, 'learning_rate': 0.001363307018331313, 'momentum': 0.7636265700459369, 'weight_decay': 0.0027334731715486305, 'leaky_relu_alpha': 0.11080990486570994}. Best is trial 0 with value: 0.4570477371068279.
[I 2026-09-25 09:54:29,270] Trial 2 finished with value: 0.3639014486574298 and parameters: {'hidden_layer': 123, 'learning_rate': 0.0031019592521037984, 'momentum': 0.13211481220152355, 'weight_decay': 1.2633197640445262e-05, 'leaky_relu_alpha': 0.0040559735830498255}. Best is trial 2 with value: 0.3639014486574298.
[I 2026-09-25 10:00:08,080] Trial 3 finished with value: 0.4500262

{'hidden_layer': 120, 'learning_rate': 0.005317746800160751, 'momentum': 0.20894003959285906, 'weight_decay': 2.9130934686171106e-06, 'leaky_relu_alpha': 0.0029763805914404997}
0.35092531902181173


[I 2026-09-25 13:39:50,793] Trial 0 finished with value: 0.4693701107610641 and parameters: {'hidden_layer': 47, 'learning_rate': 0.001734262365906389, 'momentum': 0.5047107896315642, 'weight_decay': 0.002032046171562881, 'elu_alpha': 1.3388134873784838}. Best is trial 0 with value: 0.4693701107610641.
[I 2026-09-25 13:47:39,141] Trial 1 finished with value: 0.453194784500096 and parameters: {'hidden_layer': 28, 'learning_rate': 0.0006927222559905066, 'momentum': 0.31742486722592045, 'weight_decay': 2.7946446306176854e-06, 'elu_alpha': 0.4791689934200174}. Best is trial 1 with value: 0.453194784500096.
[I 2026-09-25 13:53:07,686] Trial 2 finished with value: 0.47008506173751313 and parameters: {'hidden_layer': 96, 'learning_rate': 0.0027896345072476493, 'momentum': 0.09468942227907774, 'weight_decay': 3.285190188649579e-05, 'elu_alpha': 1.4476399735511372}. Best is trial 1 with value: 0.453194784500096.
[I 2026-09-25 13:53:16,936] Trial 3 finished with value: inf and parameters: {'hidd

{'hidden_layer': 82, 'learning_rate': 0.0004910762739354061, 'momentum': 0.9257687564245632, 'weight_decay': 1.3139548430197805e-05, 'elu_alpha': 0.1409809797548997}
0.3616198126481279


[I 2026-09-25 18:16:21,865] Trial 0 finished with value: 0.4070397459731634 and parameters: {'hidden_layer': 26, 'learning_rate': 0.00802892031446619, 'momentum': 0.06309611296163853, 'weight_decay': 8.20258637894404e-06}. Best is trial 0 with value: 0.4070397459731634.
[I 2026-09-25 18:20:58,206] Trial 1 finished with value: 0.4286748736933291 and parameters: {'hidden_layer': 74, 'learning_rate': 0.00019930292938529843, 'momentum': 0.9132897097333287, 'weight_decay': 5.658695845569574e-06}. Best is trial 0 with value: 0.4070397459731634.
[I 2026-09-25 18:26:07,019] Trial 2 finished with value: 0.5222480224578778 and parameters: {'hidden_layer': 22, 'learning_rate': 0.00027185651981566074, 'momentum': 0.11379155274446981, 'weight_decay': 7.047009018720604e-06}. Best is trial 0 with value: 0.4070397459731634.
[I 2026-09-25 18:31:22,458] Trial 3 finished with value: 0.47533748973298284 and parameters: {'hidden_layer': 15, 'learning_rate': 0.0008349020243737295, 'momentum': 0.013033356717

KeyboardInterrupt: 

In [10]:
acts = ['relu', 'leaky_relu', 'elu']
results = []
for activation in acts:
    setup = {'activation': activation}
    alpha = 0
    params = retrieve_params(f'params/california/{activation}.json')
    if activation == 'leaky_relu':
        alpha = params["leaky_relu_alpha"]
    elif activation == 'elu':
        alpha = params["elu_alpha"]
    result = cv_regression(X_pool, y_pool, out_units=1, out_act='linear', 
                    loss='mse', k=5, random_state=42, 
                    hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
                    learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'])
    for key, val in result.items():
        setup[key] = val

    results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/california/cv_validation.csv", index=False)

In [5]:
results_df = pd.read_csv("results/california/cv_validation.csv")
results_df

,activation,rmse,mape,R_squared
0,relu,0.6678,0.2625,0.6476
1,leaky_relu,0.5929,0.2447,0.7345
2,elu,0.6083,0.2558,0.7204


In [30]:
activation = 'leaky_relu'
params = retrieve_params(f'params/california/{activation}.json')
scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
model = create_regmodel(X_train.shape[1], hidden_units=params['hidden_layer'], activation=activation, alpha=params['leaky_relu_alpha'], 
                 learning_rate=params['learning_rate'], momentum=0.02, weight_decay=params['weight_decay'], loss='mse')
# params['learning_rate']
history = []
runs = 100
# runs = 200
epochs = 200 // runs

fit_kwargs = {
    "epochs": epochs,
    "batch_size": 32,
    # "validation_split": 0.15,
    # "callbacks": [early_stopping],
    "verbose": 0
}
tic = time()
for i in range(runs):
    iteration = {'cumulative epochs': epochs * (i + 1)}
    iter_hist = model.fit(
        X_train,
        y_pool,
        **fit_kwargs
    )

    train_loss = iter_hist.history['loss']
    weights = model.get_weights()

    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

    if not all(np.all(np.isfinite(w)) for w in weights):
        print(f"Model became unstable at iteration {i + 1}")
        break

    y_pred = model(X_val, training=False).numpy().ravel()

    if not np.all(np.isfinite(y_pred)):
        print(f"NaN/Inf predictions at iteration {i + 1}")
        break

    rmse = root_mean_squared_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    toc = time()

    iteration['Training loss'] = train_loss[-1]
    iteration['RMSE'] = rmse
    iteration['MAPE'] = mape
    iteration['R2-score'] = r2
    iteration['time'] = toc - tic
    history.append(iteration)
    
    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/california/passive_learning.csv", index=False)

Iteration 1: 
	0.9823712110519409
	0.5953853130340576
Iteration 1: 
	0.9823712110519409
	0.5953853130340576
Iteration 2: 
	0.5597645044326782
	0.5119470357894897
Iteration 2: 
	0.5597645044326782
	0.5119470357894897
Iteration 3: 
	0.4798978269100189
	0.4788159132003784
Iteration 3: 
	0.4798978269100189
	0.4788159132003784
Iteration 4: 
	0.4629015028476715
	0.45966559648513794
Iteration 4: 
	0.4629015028476715
	0.45966559648513794
Iteration 5: 
	0.4492672085762024
	0.4436721205711365
Iteration 5: 
	0.4492672085762024
	0.4436721205711365
Iteration 6: 
	0.4414399266242981
	0.4415772557258606
Iteration 6: 
	0.4414399266242981
	0.4415772557258606
Iteration 7: 
	0.4346224367618561
	0.44651395082473755
Iteration 7: 
	0.4346224367618561
	0.44651395082473755
Iteration 8: 
	0.42259863018989563
	0.42580193281173706
Iteration 8: 
	0.42259863018989563
	0.42580193281173706
Iteration 9: 
	0.4185628592967987
	0.4180830419063568
Iteration 9: 
	0.4185628592967987
	0.4180830419063568
Iteration 10: 
	0.41

In [13]:
activation = 'leaky_relu'
# params['leaky_relu_alpha']
# activation = 'relu'
params = retrieve_params(f'params/california/{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)

history = active_learning_reg(X_train, y_pool, X_val, y_test, random_sampling, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                               hidden_units=params['hidden_layer'], activation=activation, alpha=params['leaky_relu_alpha'],
                               learning_rate=params['learning_rate'], momentum=0.02, weight_decay=params['weight_decay'] 
)

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/california/uncertainty_sampling_reg.csv", index=False)

Iteration 1: 1651 labelled instances
	2.4452600479125977
	0.9759215116500854
	Test MAPE = 0.3952252245513445

Iteration 2: 1801 labelled instances
	0.8323790431022644
	0.8148404359817505
	Test MAPE = 0.3559932459001808

Iteration 3: 1951 labelled instances
	0.7612895965576172
	0.7220782041549683
	Test MAPE = 0.34849916298452244

Iteration 4: 2101 labelled instances
	0.7112033367156982
	0.7008821964263916
	Test MAPE = 0.3340011655520416

Iteration 5: 2251 labelled instances
	0.6672309637069702
	0.6523434519767761
	Test MAPE = 0.3197361478196392

Iteration 6: 2401 labelled instances
	0.6415755748748779
	0.6204569339752197
	Test MAPE = 0.3215394021096369

Iteration 7: 2551 labelled instances
	0.6189755797386169
	0.6044051647186279
	Test MAPE = 0.3117848759931833

Iteration 8: 2701 labelled instances
	0.6124839782714844
	0.6054306626319885
	Test MAPE = 0.3004766860416338

Iteration 9: 2851 labelled instances
	0.5938308238983154
	0.5804276466369629
	Test MAPE = 0.31098664439530777

Iteratio

In [14]:
activation = 'leaky_relu'
params = retrieve_params(f'params/california/{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
for beta in [0.9]:
    history = sasla_reg(X_train, y_pool, X_val, y_test, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                                   hidden_units=params['hidden_layer'], activation=activation, alpha=params['leaky_relu_alpha'],
                                   learning_rate=params['learning_rate'], momentum=0.02, weight_decay=params['weight_decay'], beta=beta)
    
    run_df = pd.DataFrame(history)
    run_df.round(4).to_csv(f"results/california/sasla_{beta}.csv", index=False)

Iteration 1: 16512 labelled instances
	1.019371747970581
	0.6298829317092896
	Test MAPE = 0.3016471549998825

Iteration 2: 16512 labelled instances
	0.5375019907951355
	0.5239494442939758
	Test MAPE = 0.2875212619980342

Iteration 3: 16512 labelled instances
	0.49058249592781067
	0.47891464829444885
	Test MAPE = 0.26510649454987273

Iteration 4: 16512 labelled instances
	0.4749574065208435
	0.47081661224365234
	Test MAPE = 0.263144526859005

Iteration 5: 16512 labelled instances
	0.4779776334762573
	0.463943213224411
	Test MAPE = 0.25952576806376615

Iteration 6: 16512 labelled instances
	0.47072339057922363
	0.44773900508880615
	Test MAPE = 0.25796031010732773

Iteration 7: 16512 labelled instances
	0.44342663884162903
	0.4461696147918701
	Test MAPE = 0.27143228563856653

Iteration 8: 16512 labelled instances
	0.4318358898162842
	0.42480355501174927
	Test MAPE = 0.2552850435956843

Iteration 9: 16512 labelled instances
	0.4274255037307739
	0.4187794029712677
	Test MAPE = 0.25350810011

## Concrete

In [2]:
concrete_compressive_strength = fetch_ucirepo(id=165) 
ccs_df = pd.concat(
    [concrete_compressive_strength.data.features, concrete_compressive_strength.data.targets],
    axis=1
)
ccs_df.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Concrete compressive strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [3]:
X = concrete_compressive_strength.data.features.to_numpy()
y = concrete_compressive_strength.data.targets.to_numpy()

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=51
)

In [ ]:
acts = ['relu', 'leaky_relu', 'elu']
for activation in acts:
    base_params = tune_params(X_pool, y_pool,
                              loss='mse',
                              out_units=1, out_act='linear',
                              k=5, activation=activation,
                              random_state=42,
                              n_trials=80,
                              export=True, file = f'params/concrete/{activation}.json'
    )

In [13]:
acts = ['relu', 'leaky_relu', 'elu']
results = []
for activation in acts:
    setup = {'activation': activation}
    alpha = 0
    params = retrieve_params(f'params/concrete/{activation}.json')
    if activation == 'leaky_relu':
        alpha = params["leaky_relu_alpha"]
    elif activation == 'elu':
        alpha = params["elu_alpha"]
    result = cv_regression(X_pool, y_pool, out_units=1, out_act='linear', 
                    loss='mse', k=5, random_state=42, 
                    hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
                    learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'])
    for key, val in result.items():
        setup[key] = val

    results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/concrete/cv_validation.csv", index=False)

In [4]:
results_df = pd.read_csv("results/concrete/cv_validation.csv")
results_df

,activation,rmse,mape,R_squared
0,relu,9.2752,0.2655,0.6822
1,leaky_relu,10.5160,0.3173,0.6035
2,elu,7.0351,0.1849,0.8225


In [8]:
activation = 'elu'
params = retrieve_params(f'params/concrete/{activation}.json')
scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
model = create_regmodel(X_train.shape[1], hidden_units=params['hidden_layer'], activation=activation, alpha=params['elu_alpha'], 
                 learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], loss='mse')
# params['learning_rate']
history = []
runs = 100
# runs = 200
epochs = 200 // runs

fit_kwargs = {
    "epochs": epochs,
    "batch_size": 32,
    # "validation_split": 0.15,
    # "callbacks": [early_stopping],
    "verbose": 0
}
tic = time()
for i in range(runs):
    iteration = {'cumulative epochs': epochs * (i + 1)}
    iter_hist = model.fit(
        X_train,
        y_pool,
        **fit_kwargs
    )

    train_loss = iter_hist.history['loss']
    weights = model.get_weights()

    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

    if not all(np.all(np.isfinite(w)) for w in weights):
        print(f"Model became unstable at iteration {i + 1}")
        break

    y_pred = model(X_val, training=False).numpy().ravel()

    if not np.all(np.isfinite(y_pred)):
        print(f"NaN/Inf predictions at iteration {i + 1}")
        break

    rmse = root_mean_squared_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    toc = time()

    iteration['Training loss'] = train_loss[-1]
    iteration['RMSE'] = rmse
    iteration['MAPE'] = mape
    iteration['R2-score'] = r2
    iteration['time'] = toc - tic
    history.append(iteration)
    
    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/concrete/passive_learning.csv", index=False)

Iteration 1: 
	715.452392578125
	118.73147583007812
Iteration 1: 
	715.452392578125
	118.73147583007812
Iteration 2: 
	108.20952606201172
	88.74417114257812
Iteration 2: 
	108.20952606201172
	88.74417114257812
Iteration 3: 
	81.27278900146484
	78.40422821044922
Iteration 3: 
	81.27278900146484
	78.40422821044922
Iteration 4: 
	71.33354187011719
	69.12801361083984
Iteration 4: 
	71.33354187011719
	69.12801361083984
Iteration 5: 
	74.04510498046875
	65.99049377441406
Iteration 5: 
	74.04510498046875
	65.99049377441406
Iteration 6: 
	66.22148895263672
	66.43378448486328
Iteration 6: 
	66.22148895263672
	66.43378448486328
Iteration 7: 
	68.75662231445312
	61.77617645263672
Iteration 7: 
	68.75662231445312
	61.77617645263672
Iteration 8: 
	67.7855224609375
	65.49376678466797
Iteration 8: 
	67.7855224609375
	65.49376678466797
Iteration 9: 
	62.75283432006836
	63.087310791015625
Iteration 9: 
	62.75283432006836
	63.087310791015625
Iteration 10: 
	65.11071014404297
	59.605159759521484
Iteratio

In [4]:
activation = 'elu'
params = retrieve_params(f'params/concrete/{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)

history = active_learning_reg(X_train, y_pool, X_val, y_test, random_sampling, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                               hidden_units=params['hidden_layer'], activation=activation, alpha=params['elu_alpha'],
                               learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'] 
)

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/concrete/uncertainty_sampling_reg.csv", index=False)

Iteration 1: 82 labelled instances
	1652.8074951171875
	1482.6337890625
	Test MAPE = 0.8513905685010371

Iteration 2: 89 labelled instances
	1154.8743896484375
	645.2828979492188
	Test MAPE = 0.41995340695202454

Iteration 3: 96 labelled instances
	257.98114013671875
	125.75469970703125
	Test MAPE = 0.4716047526195972

Iteration 4: 103 labelled instances
	143.87001037597656
	125.26988220214844
	Test MAPE = 0.3587848248374468

Iteration 5: 110 labelled instances
	126.94591522216797
	115.97632598876953
	Test MAPE = 0.3144570700694373

Iteration 6: 117 labelled instances
	96.29742431640625
	98.36064147949219
	Test MAPE = 0.32837733360740434

Iteration 7: 124 labelled instances
	102.82125091552734
	100.84709167480469
	Test MAPE = 0.28394330711411775

Iteration 8: 131 labelled instances
	90.04818725585938
	94.23802185058594
	Test MAPE = 0.28644058714125425

Iteration 9: 138 labelled instances
	104.96196746826172
	102.18533325195312
	Test MAPE = 0.2780566706239307

Iteration 10: 145 labelled

In [5]:
activation = 'elu'
params = retrieve_params(f'params/concrete/{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
for beta in [0.9]:
    history = sasla_reg(X_train, y_pool, X_val, y_test, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                                   hidden_units=params['hidden_layer'], activation=activation, alpha=params['elu_alpha'],
                                   learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], beta=beta)
    
    run_df = pd.DataFrame(history)
    run_df.round(4).to_csv(f"results/concrete/sasla_{beta}.csv", index=False)

Iteration 1: 824 labelled instances
	688.2462158203125
	123.20750427246094
	Test MAPE = 0.2741092705225349

Iteration 2: 824 labelled instances
	104.38902282714844
	88.89701843261719
	Test MAPE = 0.21002605618405695

Iteration 3: 824 labelled instances
	81.99990844726562
	76.89398956298828
	Test MAPE = 0.19235272940064357

Iteration 4: 824 labelled instances
	76.14723205566406
	70.67788696289062
	Test MAPE = 0.1847347545371697

Iteration 5: 824 labelled instances
	69.58319091796875
	70.58709716796875
	Test MAPE = 0.19958019366230306

Iteration 6: 824 labelled instances
	64.50582885742188
	59.121524810791016
	Test MAPE = 0.19796174347036155

Iteration 7: 824 labelled instances
	65.65623474121094
	64.92139434814453
	Test MAPE = 0.18277869745450245

Iteration 8: 824 labelled instances
	70.00354766845703
	59.065704345703125
	Test MAPE = 0.1776642086578215

Iteration 9: 824 labelled instances
	65.09345245361328
	62.507816314697266
	Test MAPE = 0.16659587813195187

Iteration 10: 824 labelled

## Seoul Bike sharing

In [6]:
seoul_bike_sharing_demand = fetch_ucirepo(id=560)
sbsd_df = pd.concat(
    [seoul_bike_sharing_demand.data.features, seoul_bike_sharing_demand.data.targets],
    axis=1
)

drop = ['Date', 'Hour', "Month", "DayOfWeek", 'Seasons']
sbsd_df['Date'] = pd.to_datetime(
    sbsd_df["Date"],
    format="%d/%m/%Y"
)
sbsd_df["Hour_sin"] = np.sin(2 * np.pi * sbsd_df["Hour"] / 24)
sbsd_df["Hour_cos"] = np.cos(2 * np.pi * sbsd_df["Hour"] / 24)

sbsd_df["Month"] = sbsd_df["Date"].dt.month
sbsd_df["DayOfWeek"] = sbsd_df["Date"].dt.dayofweek

sbsd_df["Month_sin"] = np.sin(2 * np.pi * sbsd_df["Month"] / 12)
sbsd_df["Month_cos"] = np.cos(2 * np.pi * sbsd_df["Month"] / 12)

sbsd_df["DayOfWeek_sin"] = np.sin(2 * np.pi * sbsd_df["DayOfWeek"] / 7)
sbsd_df["DayOfWeek_cos"] = np.cos(2 * np.pi * sbsd_df["DayOfWeek"] / 7)

sbsd_df["Holiday"] = (sbsd_df["Holiday"] == "Holiday").astype(int)
sbsd_df["Functioning Day"] = (sbsd_df["Functioning Day"] == "Yes").astype(int)

sbsd_df.drop(columns=drop, inplace=True)

sbsd_df.head()

,Rented Bike Count,Temperature,Humidity,Wind speed,Visibility,Dew point temperature,Solar Radiation,Rainfall,Snowfall,Holiday,Functioning Day,Hour_sin,Hour_cos,Month_sin,Month_cos,DayOfWeek_sin,DayOfWeek_cos
0,254,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,0,1,0.000000,1.000000,-2.449294e-16,1.0,-0.433884,-0.900969
1,204,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,0,1,0.258819,0.965926,-2.449294e-16,1.0,-0.433884,-0.900969
2,173,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,0,1,0.500000,0.866025,-2.449294e-16,1.0,-0.433884,-0.900969
3,107,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,0,1,0.707107,0.707107,-2.449294e-16,1.0,-0.433884,-0.900969
4,78,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,0,1,0.866025,0.500000,-2.449294e-16,1.0,-0.433884,-0.900969


In [7]:
X = sbsd_df.drop(columns=['Rented Bike Count']).to_numpy()
y = sbsd_df['Rented Bike Count'].to_numpy()

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=51
)

In [ ]:
acts = ['relu', 'leaky_relu', 'elu']
for activation in acts:
    base_params = tune_params(X_pool, y_pool,
                              loss='mse',
                              out_units=1, out_act='linear',
                              k=5, activation=activation,
                              random_state=42,
                              n_trials=80,
                              export=True, file = f'params/seoul_bikes/{activation}.json'
    )

In [19]:
acts = ['relu', 'leaky_relu', 'elu']
results = []
for activation in acts:
    setup = {'activation': activation}
    alpha = 0
    params = retrieve_params(f'params/seoul_bikes/{activation}.json')
    if activation == 'leaky_relu':
        alpha = params["leaky_relu_alpha"]
    elif activation == 'elu':
        alpha = params["elu_alpha"]
    result = cv_regression(X_pool, y_pool, out_units=1, out_act='linear', 
                    loss='mse', k=5, random_state=42, 
                    hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
                    learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'])
    for key, val in result.items():
        setup[key] = val

    results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/seoul_bikes/cv_validation.csv", index=False)

In [7]:
results_df = pd.read_csv("results/seoul_bikes/cv_validation.csv")
results_df

,activation,rmse,mape,R_squared
0,relu,241.1777,3.171544e+15,0.8603
1,leaky_relu,243.0892,3.448171e+15,0.8579
2,elu,231.1582,9.392297e+15,0.8717


In [7]:
activation = 'elu'
params = retrieve_params(f'params/seoul_bikes/{activation}.json')
scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
model = create_regmodel(X_train.shape[1], hidden_units=params['hidden_layer'], activation=activation, alpha=params['elu_alpha'], 
                 learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], loss='mse')
# params['learning_rate']
history = []
runs = 100
# runs = 200
epochs = 200 // runs

fit_kwargs = {
    "epochs": epochs,
    "batch_size": 32,
    # "validation_split": 0.15,
    # "callbacks": [early_stopping],
    "verbose": 0
}
tic = time()
for i in range(runs):
    iteration = {'cumulative epochs': epochs * (i + 1)}
    iter_hist = model.fit(
        X_train,
        y_pool,
        **fit_kwargs
    )

    train_loss = iter_hist.history['loss']
    weights = model.get_weights()

    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

    if not all(np.all(np.isfinite(w)) for w in weights):
        print(f"Model became unstable at iteration {i + 1}")
        break

    y_pred = model(X_val, training=False).numpy().ravel()

    if not np.all(np.isfinite(y_pred)):
        print(f"NaN/Inf predictions at iteration {i + 1}")
        break

    rmse = root_mean_squared_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    toc = time()

    iteration['Training loss'] = train_loss[-1]
    iteration['RMSE'] = rmse
    iteration['MAPE'] = mape
    iteration['MAE'] = mae
    iteration['R2-score'] = r2
    iteration['time'] = toc - tic
    history.append(iteration)
    
    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/seoul_bikes/passive_learning.csv", index=False)

Iteration 1: 
	271371.375
	135844.4375
Iteration 1: 
	271371.375
	135844.4375
Iteration 2: 
	114327.609375
	101420.3125
Iteration 2: 
	114327.609375
	101420.3125
Iteration 3: 
	93129.875
	86644.03125
Iteration 3: 
	93129.875
	86644.03125
Iteration 4: 
	83417.1953125
	80139.9453125
Iteration 4: 
	83417.1953125
	80139.9453125
Iteration 5: 
	78015.0703125
	73294.859375
Iteration 5: 
	78015.0703125
	73294.859375
Iteration 6: 
	72965.8125
	71452.5078125
Iteration 6: 
	72965.8125
	71452.5078125
Iteration 7: 
	70121.34375
	67074.65625
Iteration 7: 
	70121.34375
	67074.65625
Iteration 8: 
	68969.8828125
	66530.390625
Iteration 8: 
	68969.8828125
	66530.390625
Iteration 9: 
	65855.3671875
	65657.6875
Iteration 9: 
	65855.3671875
	65657.6875
Iteration 10: 
	64201.01953125
	64392.484375
Iteration 10: 
	64201.01953125
	64392.484375
Iteration 11: 
	63395.2109375
	64908.65625
Iteration 11: 
	63395.2109375
	64908.65625
Iteration 12: 
	62141.27734375
	62868.30859375
Iteration 12: 
	62141.27734375
	628

In [8]:
activation = 'elu'
params = retrieve_params(f'params/seoul_bikes/{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)

history = active_learning_reg(X_train, y_pool, X_val, y_test, random_sampling, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                               hidden_units=params['hidden_layer'], activation=activation, alpha=params['elu_alpha'],
                               learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'] 
)

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/seoul_bikes/uncertainty_sampling_reg.csv", index=False)

Iteration 1: 700 labelled instances
	904755.0
	420137.28125
	Test MAPE = 3.979191421973299e+16

Iteration 2: 763 labelled instances
	202994.15625
	191988.25
	Test MAPE = 2.3550095957950464e+16

Iteration 3: 826 labelled instances
	177775.578125
	161995.625
	Test MAPE = 1.845965991654195e+16

Iteration 4: 889 labelled instances
	152419.765625
	143047.84375
	Test MAPE = 1.7364779337252864e+16

Iteration 5: 952 labelled instances
	146792.375
	142161.59375
	Test MAPE = 1.224598792175616e+16

Iteration 6: 1015 labelled instances
	140261.328125
	139911.90625
	Test MAPE = 1.279882000596992e+16

Iteration 7: 1078 labelled instances
	133114.171875
	127994.0625
	Test MAPE = 1.5143992408670208e+16

Iteration 8: 1141 labelled instances
	124280.734375
	123023.2734375
	Test MAPE = 1.7842671758344192e+16

Iteration 9: 1204 labelled instances
	115519.7734375
	116201.796875
	Test MAPE = 1.892964959780864e+16

Iteration 10: 1267 labelled instances
	114766.0390625
	106766.078125
	Test MAPE = 1.7020618239

In [9]:
activation = 'elu'
params = retrieve_params(f'params/seoul_bikes/{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
for beta in [0.9]:
    history = sasla_reg(X_train, y_pool, X_val, y_test, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                                   hidden_units=params['hidden_layer'], activation=activation, alpha=params['elu_alpha'],
                                   learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], beta=beta)
    
    run_df = pd.DataFrame(history)
    run_df.round(4).to_csv(f"results/seoul_bikes/sasla_{beta}.csv", index=False)

Iteration 1: 7008 labelled instances
	268123.625
	138471.9375
	Test MAPE = 1.1375652527669248e+16

Iteration 2: 7000 labelled instances
	115493.578125
	103872.2734375
	Test MAPE = 1.1639756509151232e+16

Iteration 3: 7007 labelled instances
	92817.96875
	88712.84375
	Test MAPE = 1.2042258295554048e+16

Iteration 4: 7008 labelled instances
	82297.984375
	80337.3125
	Test MAPE = 1.208479779913728e+16

Iteration 5: 7008 labelled instances
	75974.8984375
	75667.75
	Test MAPE = 9411008858685440.0

Iteration 6: 7008 labelled instances
	75214.9140625
	73566.453125
	Test MAPE = 8325271191027712.0

Iteration 7: 7008 labelled instances
	71082.03125
	69653.578125
	Test MAPE = 1.0362654426136576e+16

Iteration 8: 7008 labelled instances
	68501.2109375
	68190.5
	Test MAPE = 1.1014961174151168e+16

Iteration 9: 7008 labelled instances
	67330.0625
	66090.4921875
	Test MAPE = 8714527835160576.0

Iteration 10: 7008 labelled instances
	66076.0703125
	65602.9765625
	Test MAPE = 7483689529245696.0

Iterat